### 팟빵 다운로드 1단계
* 팟빵 id 와 제목을 함수의 인자로 받는다.
* 첫번째의 8개의 에피소드만 다운로드 받고 있다.
* 저장되는 mp3 파일명을 에피소드의 타이틀로 설정함

In [ ]:
#문자열에 있는 특수문자를 제거하는 함수
def clean_text(text):
    import re
    text = text.replace("\n", "")
    cleaned_text = re.sub('[\{\}\[\]\/?.,;:|\)*~`!^\-_+<>@\#$%&\\\=\(\'\"]','', text)
    cleaned_text = cleaned_text.replace(' ','')
    return cleaned_text

In [ ]:
clean_text("김영철의 파워FM - 진짜 미국식 영어 810회 - 타일러의 진짜 미국식 영어 = Don't be so stubborn. = 억지 좀 부리지 마세요!!")

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import os

# 반란의영문법
# https://www.podbbang.com/channels/15357
referer_url = 'https://www.podbbang.com/channels/15357'

#url = 'http://www.podbbang.com/_m_api/podcasts/16524/episodes?offset=0&sort=pubdate:desc&episode_id=0&limit=8&with=summary&cache=0'
url = 'https://app-api6.podbbang.com/channels/15357/episodes?offset=0&limit=20&sort=desc&episode_id=0&focus_center=0&with=image'
res = requests.get(url)
print(res.status_code)
if res.ok:
    json_data = res.json()    
#     #{data:[{},{},{}]}
    episode_list = json_data['data']
    print(len(episode_list))
    
    for episode in episode_list:
        print('------')
        #에피소드의 제목
        title = episode['title']
        #에피소드 다운로드 url
        mp3_url = episode['media']['url']
        print(title, mp3_url)
        
        #url 값이 있으면 다운로드 받아야 함
        if mp3_url:            
            req_header = {
                'referer':referer_url,
                'user-agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/86.0.4240.111 Safari/537.36'
            }
                        
            if not os.path.isdir('mp3'):
                os.mkdir('mp3')
            
            dir_name = os.path.join('mp3','반란의영문법')
            #print(dir_name) #mp3\반란의영문법

            #해당 디렉토리가 없으면
            if not os.path.isdir(dir_name):
                #디렉토리를 생성
                os.makedirs(dir_name)
            
            #해당 url에서 파일명만 추출
            #file_name = os.path.basename(mp3_url)
            file_name = f'{clean_text(title)}.mp3'
            #생성된 디렉토리명과 파일명을 합쳐주기
            file_name = dir_name+'/'+file_name            
            #print(file_name)

            res = requests.get(mp3_url, headers=req_header)
            if res.ok:
                #response 객체에서 binary 데이터 추출
                mp3_bin = res.content                
                #binary 데이터를 local 파일로 저장
                with open(file_name, 'wb') as file:
                    file.write(mp3_bin)

In [ ]:
def podbbang_download(channel_id, channel_title):
    import requests
    from bs4 import BeautifulSoup
    from urllib.parse import urljoin
    import json
    import os

    url = f'https://app-api6.podbbang.com/channels/{channel_id}/episodes'
    #?offset=0&limit=20&sort=desc&episode_id=0&focus_center=0&with=image'
    req_param = {
        "offset":0,
        "limit":20,
        "sort":"desc",
        "episode_id":0,
        "focus_center":0,
        "with":"image"        
    }

    res = requests.get(url, params=req_param)
    print(res.status_code)
    if res.ok:
        json_data = res.json()    
        #{data:[{},{},{}]}
        episode_list = json_data['data']
        print(len(episode_list))

        for episode in episode_list:
            print('------')
            #에피소드의 제목
            title = episode['title']
            #에피소드 다운로드 url
            mp3_url = episode['media']['url']
            print(title, mp3_url)

            #url 값이 있으면 다운로드 받아야 함
            if mp3_url:            
                req_header = {
                    'referer':f'https://www.podbbang.com/channels/{channel_id}',
                    'user-agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/86.0.4240.111 Safari/537.36'
                }

                if not os.path.isdir('mp3'):
                    os.mkdir('mp3')

                dir_name = os.path.join('mp3',channel_title)

                #해당 디렉토리가 없으면
                if not os.path.isdir(dir_name):
                    #디렉토리를 생성
                    os.makedirs(dir_name)

                #해당 url에서 파일명만 추출
                #file_name = os.path.basename(mp3_url)
                clean_title = clean_text(title)
                file_name = f'{clean_title}.mp3'
                #생성된 디렉토리명과 파일명을 합쳐주기
                file_name = dir_name+'/'+file_name            
                #print(file_name)

                res = requests.get(mp3_url, headers=req_header)
                if res.ok:
                    #response 객체에서 binary 데이터 추출
                    mp3_bin = res.content                
                    #binary 데이터를 local 파일로 저장
                    with open(file_name, 'wb') as file:
                        file.write(mp3_bin)

In [ ]:
podbbang_download(15357,'반란의영문법')

### 팟빵 다운로드 2단계
* 팟빵 id 와 제목을 함수의 인자로 받는다.
* 저장되는 mp3 파일명을 에피소드의 타이틀로 설정함
* 모든 에피소드를 다운로드 받는다.
 - 에피소드를 요청할때 offset 값의 최대값이 항상 변하므로 
 - itertools의 count() 함수를 사용해 무한루프를 수행하도록 한다.
 - 무한루프를 빠져 나올수 있는 조건식을 반드시 주어야 한다.
   - json_data['data'] 길이가 0 이면 무한루프를 탈출한다.
 - 사람이 직접 다운로드 받는 것 처럼 보이게 하기 위해서 sleep time 설정한다.
   - time 이라는 내부모듈의 sleep 함수를 사용한다.

In [ ]:
def podbbang_all_download(channel_id, channel_title):
    import requests
    from bs4 import BeautifulSoup
    from urllib.parse import urljoin
    import json
    import os
    from itertools import count
    from time import sleep
    
    url = f'https://app-api6.podbbang.com/channels/{channel_id}/episodes'
    for offset in count(0):
        req_param = {
            "offset":offset,
            "limit":20,
            "sort":"desc",
            "episode_id":0,
            "focus_center":0,
            "with":"image"        
        }

        res = requests.get(url, params=req_param)
        print(res.status_code)
        if res.ok:
            json_data = res.json()    
            #{data:[{},{},{}]}
            episode_list = json_data['data']
            print(len(episode_list))

            for episode in episode_list:
                print('------')
                #에피소드의 제목
                title = episode['title']
                #에피소드 다운로드 url
                mp3_url = episode['media']['url']
                print(title, mp3_url)

                #url 값이 있으면 다운로드 받아야 함
                if mp3_url:            
                    req_header = {
                        'referer':f'https://www.podbbang.com/channels/{channel_id}',
                        'user-agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/86.0.4240.111 Safari/537.36'
                    }

                    if not os.path.isdir('mp3'):
                        os.mkdir('mp3')

                    dir_name = os.path.join('mp3',channel_title)
                    #print(dir_name) #mp3\미드처방전

                    #해당 디렉토리가 없으면
                    if not os.path.isdir(dir_name):
                        #디렉토리를 생성
                        os.makedirs(dir_name)

                    clean_title = clean_text(title)
                    file_name = f'{clean_title}.mp3'
                    #생성된 디렉토리명과 파일명을 합쳐주기
                    file_name = dir_name+'/'+file_name            
                    #print(file_name)

                    res = requests.get(mp3_url, headers=req_header)
                    if res.ok:
                        #response 객체에서 binary 데이터 추출
                        mp3_bin = res.content                
                        #binary 데이터를 local 파일로 저장
                        with open(file_name, 'wb') as file:
                            file.write(mp3_bin)
                            
                        #0.5초간 프로세스를 중지함, 기계가 아니라 사람처럼 보이게 하려고
                        sleep(0.5)

In [6]:
podbbang_all_download('1787704','미드영어')

KeyboardInterrupt: 